In [ ]:
# https://arxiv.org/pdf/2404.19497

In [2]:
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import LightCone

# q0만 측정하는 회로
qc = QuantumCircuit(3, 1)

qc.h(1)        # q1 -> q0로 영향을 줄 수 있으므로 남을 가능성이 큼
qc.x(2)        # q2는 측정 q0와 무관하므로 제거될 것
qc.cx(1, 0)    # q1이 q0에 영향을 줌
qc.x(0)        # q0 측정 결과에 직접 영향
qc.z(2)        # q2-only 게이트라 제거될 것
qc.measure(0, 0)

pm = PassManager([LightCone()])
reduced = pm.run(qc)

print("Original circuit:")
print(qc.draw())

print("\nAfter LightCone:")
print(reduced.draw())

print("\nOriginal ops:", qc.count_ops())
print("Reduced ops:", reduced.count_ops())

Original circuit:
          ┌───┐┌───┐┌─┐
q_0: ─────┤ X ├┤ X ├┤M├
     ┌───┐└─┬─┘└───┘└╥┘
q_1: ┤ H ├──■────────╫─
     ├───┤┌───┐      ║ 
q_2: ┤ X ├┤ Z ├──────╫─
     └───┘└───┘      ║ 
c: 1/════════════════╩═
                     0 

After LightCone:
          ┌───┐┌───┐┌─┐
q_0: ─────┤ X ├┤ X ├┤M├
     ┌───┐└─┬─┘└───┘└╥┘
q_1: ┤ H ├──■────────╫─
     └───┘           ║ 
q_2: ────────────────╫─
                     ║ 
c: 1/════════════════╩═
                     0 

Original ops: OrderedDict([('x', 2), ('h', 1), ('cx', 1), ('z', 1), ('measure', 1)])
Reduced ops: OrderedDict([('h', 1), ('cx', 1), ('x', 1), ('measure', 1)])


In [3]:
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import LightCone

qc = QuantumCircuit(3)

qc.h(0)
qc.cx(0, 1)
qc.x(2)      # observable Z on q0와 무관하면 제거될 것

# qubit 0의 Z observable에 대한 light cone
reduced = PassManager([LightCone(bit_terms="Z", indices=[0])]).run(qc)

print(qc.draw())
print(reduced.draw())

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
     ┌───┐└───┘
q_2: ┤ X ├─────
     └───┘     
     ┌───┐
q_0: ┤ H ├
     └───┘
q_1: ─────
          
q_2: ─────
          
